# Io dateien

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">I/O Grundlagen: Streams, Modi und Konzepte</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 7: I/O &nbsp;|&nbsp; Notebook 07a</p>
</div>
</div>

**Legende**

> **[PCAP 5.4]** Inhalt wird in der Pruefung abgefragt  
> **[Kursinhalt]** Praxiswissen fuer den Kurs, nicht pruefungsrelevant

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">PCAP 5.4</span>
1. Das Problem: Daten ueberleben das Programm nicht
</span>
</div>

Alles was bisher in Variablen gespeichert wurde, ist weg sobald das Programm endet. Ein Spielstand, eine Bestenliste, eine Konfiguration -- nichts davon bleibt erhalten.

Um Daten dauerhaft zu speichern, muss man sie auf einem **persistenten Medium** ablegen -- zum Beispiel in einer Datei auf der Festplatte. Und um Daten aus einer Datei zu lesen, muss man wissen wie Python Dateien behandelt.

Das Konzept dahinter heisst **I/O** -- Input/Output. Daten fliessen in das Programm hinein (Input) oder aus ihm heraus (Output). Eine Datei ist eine Quelle oder ein Ziel dieses Datenflusses.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">PCAP 5.4</span>
2. Streams, Handles und vordefinierte Streams
</span>
</div>

Python modelliert jeden Datenfluss als **Stream** -- einen geordneten Fluss von Daten. Ein **Handle** (auch Datei-Objekt genannt) ist das Python-Objekt das diesen Stream repraesentiert und Methoden wie `.read()` und `.write()` bereitstellt.

Drei Streams existieren immer -- noch bevor das Programm eine Datei oeffnet:

| Stream | Name | Wohin | Standard |
|--------|------|-------|----------|
| `sys.stdin` | Standard Input | Eingabe lesen | Tastatur |
| `sys.stdout` | Standard Output | Normale Ausgabe | Konsole |
| `sys.stderr` | Standard Error | Fehlermeldungen | Konsole |

In [1]:
import sys

# Die vordefinierten Streams sind Datei-Objekte -- genau wie geoeffnete Dateien
print(type(sys.stdout))   # <class '_io.TextIOWrapper'>

# print() schreibt standardmaessig nach sys.stdout
print('Hallo')                           # identisch mit:
sys.stdout.write('Hallo\n')             # direktes Schreiben in den Stream

# Fehlermeldungen gehoeren nach stderr -- nicht nach stdout
print('Das ist ein Fehler!', file=sys.stderr)

<class 'ipykernel.iostream.OutStream'>
Hallo
Hallo


Das ist ein Fehler!


<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">PCAP 5.4</span>
3. Text- vs. Binaermodus -- und alle Oeffnungsmodi
</span>
</div>

Beim Oeffnen einer Datei muss man Python mitteilen was man vorhat. Dafuer gibt es den **Modus**. Er beantwortet zwei Fragen:

- Lesen oder Schreiben?
- Text oder Binaerdaten?

| Modus | Bedeutung | Gibt zurueck |
|-------|-----------|-------------|
| `'r'` | Lesen (Standard) | `str` |
| `'w'` | Schreiben -- ueberschreibt! | `str` |
| `'a'` | Anhaengen (append) | `str` |
| `'x'` | Erstellen -- Fehler wenn vorhanden | `str` |
| `'rb'` | Lesen binaer | `bytes` |
| `'wb'` | Schreiben binaer | `bytes` |
| `'r+'` | Lesen und Schreiben | `str` |

**Textmodus** konvertiert Zeilenumbrueche automatisch (`\n` ↔ `\r\n` auf Windows) und gibt `str` zurueck. **Binaermodus** tut nichts dergleichen und gibt `bytes` zurueck -- fuer Bilder, PDFs, ZIPs.

In [ ]:
# Textmodus: Rueckgabe ist str
with open('test.txt', 'w') as f:
    f.write('Hallo Welt\n')

with open('test.txt', 'r') as f:
    inhalt = f.read()
    print(type(inhalt))    # <class 'str'>
    print(repr(inhalt))    # 'Hallo Welt\n'

# Binaermodus: Rueckgabe ist bytes
with open('test.txt', 'rb') as f:
    inhalt = f.read()
    print(type(inhalt))    # <class 'bytes'>
    print(inhalt)          # b'Hallo Welt\n'

---

**Zusammenfassung**

| Konzept | PCAP | Kernaussage |
|---------|------|-------------|
| Stream | 5.4 | Geordneter Datenfluss -- Handle ist das Python-Objekt dazu |
| stdin/stdout/stderr | 5.4 | Vordefinierte Streams in `sys` |
| Textmodus | 5.4 | `'r'`, `'w'`, `'a'` -- gibt `str`, konvertiert Zeilenumbrueche |
| Binaermodus | 5.4 | `'rb'`, `'wb'` -- gibt `bytes`, keine Konvertierung |
| Modus `'w'` | 5.4 | Loescht Dateiinhalt sofort beim Oeffnen |
| Modus `'a'` | 5.4 | Haengt an bestehenden Inhalt an |


---

**Weiter:** In Notebook **07b -- Dateien lesen und schreiben** setzen wir das in die Praxis um: `open()`, `read()`, `write()`, `readline()`, `errno` und `bytearray`.